In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys
sys.path.append("..")

In [3]:
x_train = pd.read_csv("../src/x_train.csv")
y_train = pd.read_csv("../src/y_train.csv")
y_train = y_train.squeeze()
x_test = pd.read_csv("../src/x_test.csv")
y_test = pd.read_csv("../src/y_test.csv")
y_test = y_test.squeeze()

In [4]:
x_train.drop(columns=['student_id'],inplace=True)
x_test.drop(columns=['student_id'],inplace=True)

Based on the baseline evaluation, Support Vector Machine (SVM) and Random Forest were selected for hyperparameter tuning due to their strong predictive performance and potential for further improvement.

In [5]:
# numeric columns 
num_cols = [
    "age","cgpa","internships_count","projects_count",
    "certifications_count","coding_skill_score",
    "aptitude_score","communication_skill_score",
    "logical_reasoning_score","hackathons_participated",
    "github_repos","linkedin_connections",
    "mock_interview_score","attendance_percentage",
    "backlogs","extracurricular_score","leadership_score",
    "sleep_hours","study_hours_per_day"
]

In [6]:
# text columns for OneHotEncoder
text_cols_onehot = [
    "gender","branch","volunteer_experience"
]

In [7]:
# text columns for OrdinalEncoder
text_cols_ordinal = ["college_tier"]

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
preprocessor = ColumnTransformer(transformers=[
    ('ohe',OneHotEncoder(handle_unknown='ignore'),text_cols_onehot),
    ('oe',OrdinalEncoder(),text_cols_ordinal),
    ('scaler',StandardScaler(),num_cols)
])

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [10]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

Random Forest

In [12]:
modelRF = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('rf',RandomForestClassifier())
])
params_rf = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_split': [2, 5],
    'rf__min_samples_leaf': [1, 2],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__bootstrap': [True],
    'rf__criterion': ['gini']
}
random = RandomizedSearchCV(estimator=modelRF,param_distributions=params_rf,n_iter=10,cv=3)
random.fit(x_train,y_train)
print(random.best_params_)
print(random.best_score_)

{'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 1, 'rf__max_features': 'log2', 'rf__max_depth': 10, 'rf__criterion': 'gini', 'rf__bootstrap': True}
0.5657374695514124


In [14]:
modelRFT = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('rf',RandomForestClassifier(
        n_estimators = random.best_params_['rf__n_estimators'],
        max_depth = random.best_params_['rf__max_depth'],
        min_samples_split = random.best_params_['rf__min_samples_split'],
        min_samples_leaf = random.best_params_['rf__min_samples_leaf'],
        max_features = random.best_params_['rf__max_features'],
        bootstrap = random.best_params_['rf__bootstrap'],
        criterion = random.best_params_['rf__criterion']
        ))
])
modelRFT.fit(x_train,y_train)
predictionRFT = modelRFT.predict(x_test)

In [15]:
# result of Random Forest Tuning
from sklearn.metrics import (confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score)
cm = confusion_matrix(y_test,predictionRFT)
acc_train = modelRFT.score(x_train,y_train)
acc_test = modelRFT.score(x_test,y_test)
p = precision_score(y_test,predictionRFT)
r = recall_score(y_test,predictionRFT)
f1 = f1_score(y_test,predictionRFT)
roc = roc_auc_score(y_test,modelRFT.predict_proba(x_test)[:,1])
print("train_accuracy score :",acc_train)
print("test_accuracy score :",acc_test)
print("precision :",p)
print("recall :",r)
print("f1 score :",f1)
print("roc auc score :",roc)
print("confusion matrix :",cm)

train_accuracy score : 0.6389125
test_accuracy score : 0.5626
precision : 0.5623496273692858
recall : 0.8813574910328337
f1 score : 0.6866088701010246
roc auc score : 0.5791713125324907
confusion matrix : [[1669 7458]
 [1290 9583]]


In [16]:
import joblib
joblib.dump(modelRFT,'../Models/RandomForest_Tuned.pkl')

['../Models/RandomForest_Tuned.pkl']